In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

import pandas as pd
df=pd.read_csv('real_estate_preprocessed.csv')
df.head()

X = df.drop(['house_price_inr', 'house_id'], axis=1)
y = df['house_price_inr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
lr = LinearRegression()
lr.fit(X_train,y_train)

y_pred = lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2}")

coef=lr.coef_
intercept=lr.intercept_

R² Score: 0.8267865352943391


* Coefficient and intercept

In [ ]:
print(pd.DataFrame({'features': X.columns, 'Coefficients': coef}))
print("\n\nIntercept:", intercept)

               features  Coefficients
0             area_sqft  6.183135e-01
1              bedrooms  3.841426e-02
2             bathrooms  4.149659e-02
3        location_score  2.452357e-01
4             age_years -9.020562e-17
5      distance_city_km -2.410699e-01
6         lot_size_sqft  7.777545e-02
7            has_garage  1.995160e-02
8              has_pool  4.880197e-02
9  renovation_years_ago  1.405701e-01


Intercept: -1.5197807852850174


* Applying batch gradient descent

In [ ]:
class GDRegressor:
    
    def __init__(self,learning_rate=0.01,epochs=100):
        
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        
    def fit(self,X_train,y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        
        for i in range(self.epochs):
            
            y_hat = np.dot(X_train,self.coef_) + self.intercept_
            intercept_der = -2 * np.mean(y_train - y_hat)
            self.intercept_ = self.intercept_ - (self.lr * intercept_der)
            
            coef_der = -2 * np.dot((y_train - y_hat),X_train)/X_train.shape[0]
            self.coef_ = self.coef_ - (self.lr * coef_der)
        
        print(self.intercept_,self.coef_)
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_

In [32]:
bgd=GDRegressor(epochs=1000,learning_rate=0.01)
bgd.fit(X_train, y_train)
bgd_pred=bgd.predict(X_test)
r2_bgd = r2_score(y_test, bgd_pred)

-1.093101281252051 [ 6.77990353e-01 -2.63947479e-03  2.40470937e-02  2.22125926e-01
 -9.31012813e-02 -2.89350830e-01  8.91696874e-02  3.83467979e-04
  2.75060262e-01  2.41089274e-01]


* Coefficient and intercept

In [39]:
print(pd.DataFrame({'Coefficients': bgd.coef_}, index=X.columns))
print()
print("Intercept:", bgd.intercept_)

                      Coefficients
area_sqft                 0.677990
bedrooms                 -0.002639
bathrooms                 0.024047
location_score            0.222126
age_years                -0.093101
distance_city_km         -0.289351
lot_size_sqft             0.089170
has_garage                0.000383
has_pool                  0.275060
renovation_years_ago      0.241089

Intercept: -1.093101281252051


* R2 score

In [33]:
r2_bgd

0.8163457522597127